# Каракалпакский OCR: распознавание → отчёт → проверка глазами → переделка

Порядок: запускай ячейки сверху вниз (кнопка ▶ слева). После закрытия блокнота всё сохранено на Google Диске.

1. **Подготовка**: каждый раз после открытия блокнота.
2. **Настройки**: выбери книгу и режим.
3. **Распознавание**: программа сама берёт страницы по очереди и отправляет в модель. Дневной лимит остановит её сам: завтра нажми ту же ячейку, она продолжит.
4. **Отчёт**: программа сама находит подозрительные страницы (латиница, странные буквы, пропущенные статьи, пустые страницы).
5. **Проверка глазами**: слева картинка страницы, справа распознанный текст. Нажми «Всё верно», «Есть ошибки» или «Переделать».
6. **Переделка**: страницы с отметкой «Переделать» уходят на повторное распознавание.
7. **Итоги**: оценка точности по случайной выборке (для README).

In [ ]:
#@title 1. Подготовка (запускай каждый раз после открытия блокнота)
!pip install -q google-genai pymupdf ipywidgets pandas
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, importlib
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
ROOT = "/content/drive/MyDrive/OCR"
sys.path.insert(0, ROOT)
import ocr_qa
importlib.reload(ocr_qa)
print("Готово. Файлы программы найдены:", os.path.exists(ROOT + "/ocr_dict.py"), os.path.exists(ROOT + "/ocr_qa.py"))

In [ ]:
#@title 2. Настройки (измени значения и нажми ▶)
PDF_PATH = "/content/drive/MyDrive/OCR/vol7.pdf"  #@param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/OCR/out_vol7_lite35"  #@param {type:"string"}
MODE = "dict"  #@param ["dict", "text"]
MODEL = "gemini-3.5-flash-lite"  #@param {type:"string"}
REDO_MODEL = "gemini-3-flash-preview"  #@param {type:"string"}
REVIEW_HOW = "both"  #@param ["flagged", "random", "both"]
N_PAGES = 12  #@param {type:"integer"}
VOCAB_CSV = ""  #@param {type:"string"}
DECISIONS = RESULT_DIR.rstrip("/") + "_review.csv"
REPORT = RESULT_DIR.rstrip("/") + "_qa_report.csv"
print("Книга:", PDF_PATH, "| режим:", MODE, "| результаты:", RESULT_DIR)

In [ ]:
#@title 3. Распознавание (продолжит с места остановки)
ocr_qa.run_ocr(PDF_PATH, RESULT_DIR, MODE, MODEL)

In [ ]:
#@title 4. Автоматический отчёт: какие страницы подозрительные
import pandas as pd
pd.set_option("display.max_colwidth", 90)
rows = ocr_qa.build_report(RESULT_DIR, PDF_PATH, MODE, VOCAB_CSV or None, save_to=REPORT)
print("Страниц в отчёте:", len(rows), "| с подозрениями:", sum(r["score"] >= 2 for r in rows), "| полный отчёт сохранён:", REPORT)
pd.DataFrame(rows).head(25)

In [ ]:
#@title 5. Проверка глазами (слева картинка, справа текст; найденные подозрительные слова выделены)
labels = ocr_qa.pick_labels(rows, REVIEW_HOW, N_PAGES)
ui = ocr_qa.review(PDF_PATH, RESULT_DIR, list(labels), DECISIONS, MODE, rows, sample=labels)

In [ ]:
#@title 6. Переделать страницы с отметкой «Переделать»
pages = ocr_qa.redo_pages(DECISIONS)
print("На переделку:", pages)
if pages:
    ocr_qa.run_ocr(PDF_PATH, RESULT_DIR, MODE, REDO_MODEL, pages=pages, force=True, sleep=13)
    print("Отмечено как переделанные:", ocr_qa.mark_redone(DECISIONS, RESULT_DIR))
    print("Теперь запусти ячейку 4 ещё раз, чтобы обновить отчёт.")

In [ ]:
#@title 7. Итоги: оценка точности по случайной выборке
ocr_qa.accuracy_summary(DECISIONS)